# Explorando as características financeiras do dataset
**Dataset:** default_of_credit_card_clients

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Carregar o dataset
df = pd.read_excel('default_of_credit_card_clients__courseware_version_1_21_19.xls', engine='xlrd')

print('Shape:', df.shape)
df.head()

## Exercício 1 (2pts)
Criar listas com os nomes das características financeiras restantes.

In [ ]:
# Lista com as características de valor da fatura (bill amount)
bill_feats = ['BILL_AMT1', 'BILL_AMT2', 'BILL_AMT3', 'BILL_AMT4', 'BILL_AMT5', 'BILL_AMT6']

# Lista com as características de valor do pagamento (pay amount)
pay_feats = ['PAY_AMT1', 'PAY_AMT2', 'PAY_AMT3', 'PAY_AMT4', 'PAY_AMT5', 'PAY_AMT6']

print('Características de fatura:', bill_feats)
print('Características de pagamento:', pay_feats)

## Exercício 2 (3pts)
Usar `.describe()` para examinar as sínteses estatísticas das características de valor da fatura e refletir sobre os resultados.

In [ ]:
df[bill_feats].describe()

**Reflexão:** As características de valor da fatura (`BILL_AMT1` a `BILL_AMT6`) apresentam valores muito díspares: enquanto a média gira em torno de 50.000 a 60.000, os valores mínimos são negativos (possíveis créditos ou estornos) e os máximos ultrapassam 900.000. O desvio padrão elevado (por volta de 70.000 a 80.000) indica uma distribuição bastante assimétrica, com poucos clientes com faturas muito altas puxando a média para cima. Isso sugere que a maioria dos clientes possui faturas relativamente baixas, e a distribuição não segue uma curva normal — o que justifica o uso do logaritmo para visualização.

## Exercício 3 (4pts)
Visualizar as características de valor da fatura em uma grade 2x3 de histogramas com 20 bins.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(12, 7))

for i, col in enumerate(bill_feats):
    ax = axes[i // 3][i % 3]
    df[col].hist(bins=20, ax=ax, color='steelblue', edgecolor='white')
    ax.set_title(col)
    ax.set_xlabel('Valor')
    ax.set_ylabel('Frequência')

plt.suptitle('Distribuição das características de valor da fatura', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## Exercício 4 (4pts)
Obter o resumo de `.describe()` para as características de valor do pagamento e refletir.

In [ ]:
df[pay_feats].describe()

**Reflexão:** As características de valor do pagamento (`PAY_AMT1` a `PAY_AMT6`) têm média entre 4.000 e 6.000, mas mediana (50%) muito próxima de zero em alguns meses — indicando que uma parcela significativa dos clientes não realizou pagamentos em determinados períodos. O valor mínimo é 0 (sem pagamentos negativos), e os máximos são extremamente altos (acima de 1 milhão), evidenciando forte assimetria positiva. A grande diferença entre média e mediana confirma que a distribuição é dominada por muitos valores pequenos e poucos valores muito grandes — o que torna o logaritmo uma ferramenta útil para visualizar esse tipo de dado.

## Exercício 5 (4pts)
Plotar histograma das características de pagamento com grade 2x3 e rotação nos rótulos do eixo x.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(12, 7))

for i, col in enumerate(pay_feats):
    ax = axes[i // 3][i % 3]
    df[col].hist(bins=20, ax=ax, color='coral', edgecolor='white')
    ax.set_title(col)
    ax.set_xlabel('Valor')
    ax.set_ylabel('Frequência')
    ax.tick_params(axis='x', rotation=45)  # xrot=45

plt.suptitle('Distribuição das características de valor do pagamento', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

**Observação:** Os histogramas confirmam a forte concentração dos pagamentos próximos a zero, com uma barra muito alta na primeira classe e valores decrescendo rapidamente. Isso reforça que a maioria dos clientes faz pagamentos pequenos ou não paga, enquanto uma minoria realiza pagamentos muito altos.

## Exercício 6 (4pts)
Usar máscara booleana para ver quantos valores de pagamento são exatamente iguais a 0.

In [ ]:
# Máscara booleana: True onde o valor é igual a 0
zero_mask = df[pay_feats] == 0

# Contagem de zeros por coluna
print('Quantidade de zeros por coluna de pagamento:')
print(zero_mask.sum())

print('\nPercentual de zeros por coluna:')
print((zero_mask.sum() / len(df) * 100).round(2).astype(str) + '%')

**Reflexão:** Faz muito sentido! Os histogramas do Exercício 5 mostravam uma barra enorme no valor 0, e agora confirmamos numericamente: em cada mês, entre 20% e 30% dos clientes tiveram pagamento igual a zero. Isso explica a forma extremamente concentrada dos histogramas — a maioria dos valores é 0, e os demais se distribuem à direita.

## Exercício 7 (4pts)
Ignorar os pagamentos iguais a 0 e plotar histogramas com transformação logarítmica (`np.log10`) usando `.apply()`.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(12, 7))

for i, col in enumerate(pay_feats):
    ax = axes[i // 3][i % 3]
    
    # Filtrar os valores diferentes de zero usando a máscara booleana
    non_zero = df[col][df[col] != 0]
    
    # Aplicar log10 com .apply()
    non_zero_log = non_zero.apply(np.log10)
    
    non_zero_log.hist(bins=20, ax=ax, color='mediumseagreen', edgecolor='white')
    ax.set_title(col)
    ax.set_xlabel('log10(Valor)')
    ax.set_ylabel('Frequência')
    ax.tick_params(axis='x', rotation=45)

plt.suptitle('Histogramas de transformações logarítmicas dos pagamentos ≠ 0', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

**Reflexão:** Após remover os zeros e aplicar a transformação logarítmica, as distribuições ficam muito mais legíveis e próximas de uma curva normal (distribuição aproximadamente simétrica). Isso mostra que os valores de pagamento seguem uma **distribuição log-normal**: quando transformados pelo logaritmo, revelam um padrão mais compreensível. A maioria dos pagamentos não-nulos se concentra entre log10(3) ≈ 1.000 e log10(5) ≈ 100.000, com pico em torno de log10(4) ≈ 10.000.